In [1]:
%cd ../..

/path/to/repo/Documents/my-coles-gnn-experimetns/scenario_gender


In [2]:
import os

In [3]:
PRETRAINED_MODELS_ROOT = 'models/pretrained_gnn_models'

In [4]:
model_dirs = os.listdir(PRETRAINED_MODELS_ROOT)
model_dirs

['wl-0.5_gnn-GAT_res-True_wd-0.1',
 'wl-0.5_gnn-GraphSAGE_res-True_wd-0.1',
 '.DS_Store',
 'wl-0.5_gnn-GraphSAGE_res-True_wd-0',
 'wl-0.5_gnn-GraphSAGE_res-True_wd-0.01']

In [5]:
EPOCHES = list(range(50, 250, 50))
EPOCHES

[50, 100, 150, 200]

In [7]:
MAX_EPOCHES=40

for model_dir in model_dirs:
    for epoch in EPOCHES:

        f_name = f"{epoch}.pt"

        embeddings_path = os.path.join(PRETRAINED_MODELS_ROOT, model_dir, f_name)

        experiment_name = f"coles_gnn__pretrained_{model_dir}__pretrain_epoches_{epoch}__epoches_{MAX_EPOCHES}"

        !PYTHONPATH=.. python -m ptls.pl_train_module \
        --config-dir conf --config-name mles_params \
        data_module.train_data.splitter.split_count=2 \
        data_module.valid_data.splitter.split_count=2 \
        pl_module.validation_metric.K=1 \
        pl_module.lr_scheduler_partial.step_size=60 \
        model_path="models/{experiment_name}.p" \
        logger_name="{experiment_name}" \
        data_module.train_batch_size=64 \
        data_module.train_num_workers=4 \
        data_module.valid_batch_size=64 \
        data_module.valid_num_workers=4  \
        pl_module.seq_encoder.trx_encoder.custom_embeddings.mcc_code.embeddings_path="{embeddings_path}" \ 
        trainer.max_epochs=40
        device="cpu"

        !PYTHONPATH=.. python -m ptls.pl_inference    \
        model_path="models/{experiment_name}_model.p" \
        embed_file_name="{experiment_name}_embeddings" \
        inference.batch_size=16 \
        --config-dir conf --config-name coles_gnn_pretrained_params \
    

LexerNoViableAltException: \
                           ^
See https://hydra.cc/docs/1.2/advanced/override_grammar/basic for details

Set the environment variable HYDRA_FULL_ERROR=1 for a complete stack trace.


NameError: name 'trainer' is not defined

In [5]:
# # Compare
# !rm results/scenario_gender_2024_research.txt
# # rm -r conf/embeddings_validation.work/
# !python -m embeddings_validation \
#     --config-dir conf --config-name embeddings_validation__2024_research_weighted +workers=10 +total_cpu_count=4

In [8]:
dir_to_epoches = {
    "wl-0.5_gnn-GAT_res-True_wd-0": [15, 50, 100, 150, 200],
    "wl-0.5_gnn-GAT_res-True_wd-0.1": [25, 50, 75, 100, 150, 200],
    "wl-0.5_gnn-GAT_res-True_wd-0.01": [25, 50, 100, 150, 200],
    "wl-0.5_gnn-GraphSAGE_res-False_wd-0": [50, 75, 100, 150, 200],
    "wl-0.5_gnn-GraphSAGE_res-True_wd-0": [75, 100, 150, 200],
    "wl-0.5_gnn-GraphSAGE_res-True_wd-0.1": [50, 75, 100, 150, 200],
    "wl-0.5_gnn-GraphSAGE_res-True_wd-0.01": [25, 50, 75, 100, 150, 200],
}

In [17]:
for k, v in dir_to_epoches.items():
    epoches_str = " ".join([str(x) for x in v])
    print(f'model_epoch_map[\"{k}\"]=\"{epoches_str}\"')

model_epoch_map["wl-0.5_gnn-GAT_res-True_wd-0"]="15 50 100 150 200"
model_epoch_map["wl-0.5_gnn-GAT_res-True_wd-0.1"]="25 50 75 100 150 200"
model_epoch_map["wl-0.5_gnn-GAT_res-True_wd-0.01"]="25 50 100 150 200"
model_epoch_map["wl-0.5_gnn-GraphSAGE_res-False_wd-0"]="50 75 100 150 200"
model_epoch_map["wl-0.5_gnn-GraphSAGE_res-True_wd-0"]="75 100 150 200"
model_epoch_map["wl-0.5_gnn-GraphSAGE_res-True_wd-0.1"]="50 75 100 150 200"
model_epoch_map["wl-0.5_gnn-GraphSAGE_res-True_wd-0.01"]="25 50 75 100 150 200"


In [11]:
from collections import defaultdict
import os

CHECKPOINTS_DIR = "../../checkpoints"

dir_to_epoches = {
    "wl-0.5_gnn-GAT_res-True_wd-0": [15, 50, 100, 150, 200],
    "wl-0.5_gnn-GAT_res-True_wd-0.1": [25, 50, 75, 100, 150, 200],
    "wl-0.5_gnn-GAT_res-True_wd-0.01": [25, 50, 100, 150, 200],
    "wl-0.5_gnn-GraphSAGE_res-False_wd-0": [50, 75, 100, 150, 200],
    "wl-0.5_gnn-GraphSAGE_res-True_wd-0": [75, 100, 150, 200],
    "wl-0.5_gnn-GraphSAGE_res-True_wd-0.1": [50, 75, 100, 150, 200],
    "wl-0.5_gnn-GraphSAGE_res-True_wd-0.01": [25, 50, 75, 100, 150, 200],
}

dir_to_unlearned_epoches = defaultdict(list)

correct_ckpts_lst = ['epoch=9.ckpt', 'epoch=29.ckpt', 'epoch=39.ckpt', 'epoch=19.ckpt']

for model_name, epoches in dir_to_epoches.items():
    for epoch in epoches:
        ckpt_dirname = f'coles_gnn__pretrained_{model_name}__pretrain_epoches_{epoch}__epoches_40'
        ckpt_dir_path = os.path.join(CHECKPOINTS_DIR, ckpt_dirname)
        if os.path.exists(ckpt_dir_path) and os.listdir(ckpt_dir_path) == correct_ckpts_lst:
            continue
        dir_to_unlearned_epoches[model_name].append(epoch)

dict(dir_to_unlearned_epoches)

{'wl-0.5_gnn-GAT_res-True_wd-0.01': [200],
 'wl-0.5_gnn-GraphSAGE_res-True_wd-0': [75, 100, 150, 200],
 'wl-0.5_gnn-GraphSAGE_res-True_wd-0.1': [50, 75, 100, 150, 200],
 'wl-0.5_gnn-GraphSAGE_res-True_wd-0.01': [25, 50, 75, 100, 150, 200]}

In [9]:
dir_to_unlearned_epoches

defaultdict(list,
            {'wl-0.5_gnn-GAT_res-True_wd-0.01': [200],
             'wl-0.5_gnn-GraphSAGE_res-True_wd-0': [75, 100, 150, 200],
             'wl-0.5_gnn-GraphSAGE_res-True_wd-0.1': [50, 75, 100, 150, 200],
             'wl-0.5_gnn-GraphSAGE_res-True_wd-0.01': [25,
              50,
              75,
              100,
              150,
              200]})

In [12]:
os.path.exists('../../data/coles_gnn__pretrained_wl-0.5_gnn-GAT_res-True_wd-0__pretrain_epoches_15__epoches_40_embeddings.pickle')

False

In [13]:
os.listdir('../../data/')

['transactions.csv',
 '.spark_local_dir',
 'coles_gnn_weighted__w_pred_bce__no_orig__alpha_0.5__gamma_0.5_embeddings___accidently.pickle',
 'coles_gnn_weighted__w_pred_mse__no_orig_embeddings.pickle',
 'mles_model_small_batch_2_embeddings__40_epoches.pickle',
 'coles_gnn_model_2_embeddings__backup_2.pickle',
 'coles_gnn__g_0_1__no_orig_emb__sample_2__GAT__embeddings.pickle',
 'test_trx.parquet',
 'coles_gnn_weighted__w_pred_mse__has_orig.pickle',
 'coles_gnn__g_0_5__has_orig_emb__sample_2__embeddings__graph_sage.pickle',
 'coles_gnn_model_2_embeddings.pickle',
 'coles_gnn_weighted__w_pred_bce__has_orig.pickle',
 'coles_avg_pool_model_embeddings.pickle',
 'coles_gnn_weighted__w_pred_mse__no_orig__alpha_0.5__gamma_0.5_embeddings___accidently.pickle',
 'coles_gnn__g_0_9__has_orig_emb__sample_2__embeddings__graph_sage.pickle',
 'coles_gnn_weighted__w_pred_mse__has_orig__alpha_0.5__gamma_0.5_embeddings___accidently.pickle',
 'coles_gnn_weighted__w_pred_mse__no_orig__alpha_0.5__gamma_0.5__co

In [40]:
ls

bin/                          make_graph.py
checkpoints/                  models/
checks.ipynb                  notebooks/
conf/                         outputs/
data/                         pl_inference.py
distribution_target.py        pl_inference_with_client_id.py
embeddings_validation.work/   pl_train_module.py
lightning_logs/               README.md
make_dataset_help_no_ptls.py  results/
make_dataset.py


In [44]:
from collections import defaultdict
import os

EMBEDS_DIR = "./data"

dir_to_epoches = {
    # "wl-0.5_gnn-GAT_res-True_wd-0": [15, 50, 100, 150, 200],
    # "wl-0.5_gnn-GAT_res-True_wd-0.1": [25, 50, 75, 100, 150, 200],
    # "wl-0.5_gnn-GAT_res-True_wd-0.01": [25, 50, 100, 150, 200],
    "wl-0.5_gnn-GraphSAGE_res-False_wd-0": [50, 75, 100, 150, 200],
    "wl-0.5_gnn-GraphSAGE_res-True_wd-0": [75, 100, 150, 200],
    "wl-0.5_gnn-GraphSAGE_res-True_wd-0.1": [50, 75, 100, 150, 200],
    "wl-0.5_gnn-GraphSAGE_res-True_wd-0.01": [25, 50, 75, 100, 150, 200],
}

dir_to_unlearned_epoches = defaultdict(list)

correct_ckpts_lst = ['epoch=9.ckpt', 'epoch=29.ckpt', 'epoch=39.ckpt', 'epoch=19.ckpt']

for model_name, epoches in dir_to_epoches.items():
    for epoch in epoches:
        embed_fname = f'coles_gnn__pretrained_unfreeze_after_20_epoches__{model_name}__pretrain_epoches_{epoch}__epoches_40_embeddings.pickle'
        embed_f_path = os.path.join(EMBEDS_DIR, embed_fname)
        if os.path.exists(embed_f_path):
            continue
        dir_to_unlearned_epoches[model_name].append(epoch)

dict(dir_to_unlearned_epoches)



{'wl-0.5_gnn-GraphSAGE_res-True_wd-0.1': [150, 200]}

In [ ]:
dir_to_epoches = {
    # "wl-0.5_gnn-GAT_res-True_wd-0": [15, 50, 100, 150, 200],
    # "wl-0.5_gnn-GAT_res-True_wd-0.1": [25, 50, 75, 100, 150, 200],
    # "wl-0.5_gnn-GAT_res-True_wd-0.01": [25, 50, 100, 150, 200],
    "wl-0.5_gnn-GraphSAGE_res-False_wd-0": [50, 75, 100, 150, 200],
    "wl-0.5_gnn-GraphSAGE_res-True_wd-0": [75, 100, 150, 200],
    "wl-0.5_gnn-GraphSAGE_res-True_wd-0.1": [50, 75, 100, 150, 200],
    "wl-0.5_gnn-GraphSAGE_res-True_wd-0.01": [25, 50, 75, 100, 150, 200],
}

for 

In [22]:
names = [
    "coles_gnn__pretrained_unfreeze_after_{n_epoches_before_unfreeze}_epoches__wl-0.5_gnn-GraphSAGE_res-True_wd-0__pretrain_epoches_100__epoches_40",
    # "coles_gnn__pretrained_unfreeze_after_{n_epoches_before_unfreeze}_epoches__wl-0.5_gnn-GAT_res-True_wd-0.01__pretrain_epoches_100__epoches_40",
    "coles_gnn__pretrained_unfreeze_after_{n_epoches_before_unfreeze}_epoches__wl-0.5_gnn-GraphSAGE_res-True_wd-0__pretrain_epoches_75__epoches_40",
    # "coles_gnn__pretrained_unfreeze_after_{n_epoches_before_unfreeze}_epoches__wl-0.5_gnn-GAT_res-True_wd-0.01__pretrain_epoches_25__epoches_40",
    # "coles_gnn__pretrained_frozen__wl-0.5_gnn-GAT_res-True_wd-0__pretrain_epoches_15__epoches_40",
    # "coles_gnn__pretrained_frozen__wl-0.5_gnn-GAT_res-True_wd-0.01__pretrain_epoches_50__epoches_40",
    # "coles_gnn__pretrained_frozen__wl-0.5_gnn-GAT_res-True_wd-0.01__pretrain_epoches_200__epoches_40",
    "coles_gnn__pretrained_unfreeze_after_{n_epoches_before_unfreeze}_epoches__wl-0.5_gnn-GraphSAGE_res-True_wd-0__pretrain_epoches_150__epoches_40"
]

In [20]:
config_item_template = """  {experiment_name}:
    enabled: true
    read_params: 
      file_name: ${{hydra:runtime.cwd}}/data/{experiment_name}_embeddings.pickle
    target_options: {{}}
"""
for name in names:
  print(config_item_template.format(experiment_name=name))


  coles_gnn__pretrained_frozen__wl-0.5_gnn-GraphSAGE_res-True_wd-0__pretrain_epoches_100__epoches_40:
    enabled: true
    read_params: 
      file_name: ${hydra:runtime.cwd}/data/coles_gnn__pretrained_frozen__wl-0.5_gnn-GraphSAGE_res-True_wd-0__pretrain_epoches_100__epoches_40_embeddings.pickle
    target_options: {}

  coles_gnn__pretrained_frozen__wl-0.5_gnn-GAT_res-True_wd-0.01__pretrain_epoches_100__epoches_40:
    enabled: true
    read_params: 
      file_name: ${hydra:runtime.cwd}/data/coles_gnn__pretrained_frozen__wl-0.5_gnn-GAT_res-True_wd-0.01__pretrain_epoches_100__epoches_40_embeddings.pickle
    target_options: {}

  coles_gnn__pretrained_frozen__wl-0.5_gnn-GraphSAGE_res-True_wd-0__pretrain_epoches_75__epoches_40:
    enabled: true
    read_params: 
      file_name: ${hydra:runtime.cwd}/data/coles_gnn__pretrained_frozen__wl-0.5_gnn-GraphSAGE_res-True_wd-0__pretrain_epoches_75__epoches_40_embeddings.pickle
    target_options: {}

  coles_gnn__pretrained_frozen__wl-0.5_gnn

/path/to/repo/Documents/my-coles-gnn-experimetns/scenario_gender


In [46]:
from collections import defaultdict
import os

EMBEDS_DIR = "data"

dir_to_epoches = {
    # "wl-0.5_gnn-GAT_res-True_wd-0": [15, 50, 100, 150, 200],
    # "wl-0.5_gnn-GAT_res-True_wd-0.1": [25, 50, 75, 100, 150, 200],
    # "wl-0.5_gnn-GAT_res-True_wd-0.01": [25, 50, 100, 150, 200],
    "wl-0.5_gnn-GraphSAGE_res-False_wd-0": [50, 75, 100, 150, 200],
    "wl-0.5_gnn-GraphSAGE_res-True_wd-0": [75, 100, 150, 200],
    "wl-0.5_gnn-GraphSAGE_res-True_wd-0.1": [50, 75, 100, 150, 200],
    "wl-0.5_gnn-GraphSAGE_res-True_wd-0.01": [25, 50, 75, 100, 150, 200],
}


config_item_template = """  {experiment_name}:
    enabled: true
    read_params: 
      file_name: ${{hydra:runtime.cwd}}/data/{experiment_name}_embeddings.pickle
    target_options: {{}}
"""

config_item_template_commented = """#  {experiment_name}:
#    enabled: true
#    read_params: 
#      file_name: ${{hydra:runtime.cwd}}/data/{experiment_name}_embeddings.pickle
#    target_options: {{}}
"""


n_epoches_before_unfreeze = 10

success_cnt = 0

model_prefix = f"coles_gnn__pretrained_unfreeze_after_{n_epoches_before_unfreeze}_epoches__"

for model_name, epoches in dir_to_epoches.items():
    for epoch in epoches:
        
        experiment_name = f'{model_prefix}{model_name}__pretrain_epoches_{epoch}__epoches_40'
        if os.path.exists(f'{EMBEDS_DIR}/{experiment_name}_embeddings.pickle'):
            print(config_item_template.format(experiment_name=experiment_name))
        else:
            print(config_item_template_commented.format(experiment_name=experiment_name))
        
            



  coles_gnn__pretrained_unfreeze_after_10_epoches__wl-0.5_gnn-GraphSAGE_res-False_wd-0__pretrain_epoches_50__epoches_40:
    enabled: true
    read_params: 
      file_name: ${hydra:runtime.cwd}/data/coles_gnn__pretrained_unfreeze_after_10_epoches__wl-0.5_gnn-GraphSAGE_res-False_wd-0__pretrain_epoches_50__epoches_40_embeddings.pickle
    target_options: {}

  coles_gnn__pretrained_unfreeze_after_10_epoches__wl-0.5_gnn-GraphSAGE_res-False_wd-0__pretrain_epoches_75__epoches_40:
    enabled: true
    read_params: 
      file_name: ${hydra:runtime.cwd}/data/coles_gnn__pretrained_unfreeze_after_10_epoches__wl-0.5_gnn-GraphSAGE_res-False_wd-0__pretrain_epoches_75__epoches_40_embeddings.pickle
    target_options: {}

  coles_gnn__pretrained_unfreeze_after_10_epoches__wl-0.5_gnn-GraphSAGE_res-False_wd-0__pretrain_epoches_100__epoches_40:
    enabled: true
    read_params: 
      file_name: ${hydra:runtime.cwd}/data/coles_gnn__pretrained_unfreeze_after_10_epoches__wl-0.5_gnn-GraphSAGE_res-False

In [4]:
config_item_template = """  {experiment_name}:
    enabled: true
    read_params: 
      file_name: ${{hydra:runtime.cwd}}/data/{experiment_name}_embeddings.pickle
    target_options: {{}}
"""

for ckpt_n in range(9, 150, 10):
    experiment_name = f"coles_gnn__pretrained_wl-0.5_gnn-GraphSAGE_res-True_wd-0__pretrain_epoches_100__epoches_150__ckpt_epoch_{ckpt_n}"
    print(config_item_template.format(experiment_name=experiment_name))

  coles_gnn__pretrained_wl-0.5_gnn-GraphSAGE_res-True_wd-0__pretrain_epoches_100__epoches_150__ckpt_epoch_9:
    enabled: true
    read_params: 
      file_name: ${hydra:runtime.cwd}/data/coles_gnn__pretrained_wl-0.5_gnn-GraphSAGE_res-True_wd-0__pretrain_epoches_100__epoches_150__ckpt_epoch_9_embeddings.pickle
    target_options: {}

  coles_gnn__pretrained_wl-0.5_gnn-GraphSAGE_res-True_wd-0__pretrain_epoches_100__epoches_150__ckpt_epoch_19:
    enabled: true
    read_params: 
      file_name: ${hydra:runtime.cwd}/data/coles_gnn__pretrained_wl-0.5_gnn-GraphSAGE_res-True_wd-0__pretrain_epoches_100__epoches_150__ckpt_epoch_19_embeddings.pickle
    target_options: {}

  coles_gnn__pretrained_wl-0.5_gnn-GraphSAGE_res-True_wd-0__pretrain_epoches_100__epoches_150__ckpt_epoch_29:
    enabled: true
    read_params: 
      file_name: ${hydra:runtime.cwd}/data/coles_gnn__pretrained_wl-0.5_gnn-GraphSAGE_res-True_wd-0__pretrain_epoches_100__epoches_150__ckpt_epoch_29_embeddings.pickle
    target_o